# Phase 4 · Step 3 — Prompt v3 (refuse-if-not-grounded) + v1/v2/v3 comparison

v3 adds one more instruction on top of v2: refuse explicitly instead of
guessing when the context doesn't cover the question. Runs v1/v2/v3 on
two questions — one the retrieved chunks cover, one they don't — to see
whether v3 actually refuses where v1/v2 would guess. See
`learning/06_prompts/03_prompt_v3.md` for the why behind this design.

## Reload, re-chunk, re-embed, upsert

Same setup as notebooks 08/09 — rebuilds the hybrid-search collection from
scratch so this notebook is self-contained.

In [1]:
import logging

logging.getLogger("numexpr").setLevel(logging.WARNING)

from pathlib import Path

from generate_sample_pdf import generate
from llama_index.core.node_parser import SentenceSplitter
from llama_index.readers.file import PDFReader
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, Modifier, SparseVectorParams, VectorParams

COLLECTION_NAME = "papermind_chunks"
STORAGE_PATH = Path("data/qdrant")
DENSE_MODEL = "BAAI/bge-small-en-v1.5"
SPARSE_MODEL = "Qdrant/bm25"
RERANK_MODEL = "Xenova/ms-marco-MiniLM-L-6-v2"

pdf_path = generate()
documents = PDFReader().load_data(file=pdf_path)
splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)
nodes = splitter.get_nodes_from_documents(documents)

client = QdrantClient(path=str(STORAGE_PATH))
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)
client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={"dense": VectorParams(size=384, distance=Distance.COSINE)},
    sparse_vectors_config={"bm25": SparseVectorParams(modifier=Modifier.IDF)},
)

print(f"Produced {len(nodes)} chunks from {len(documents)} pages")

Produced 4 chunks from 4 pages


In [2]:
from uuid import NAMESPACE_URL, uuid5

from fastembed import SparseTextEmbedding, TextEmbedding
from qdrant_client.models import PointStruct, SparseVector

texts = [node.get_content() for node in nodes]

dense_embedder = TextEmbedding(model_name=DENSE_MODEL)
sparse_embedder = SparseTextEmbedding(model_name=SPARSE_MODEL)

dense_vectors = list(dense_embedder.embed(texts))
sparse_vectors = list(sparse_embedder.embed(texts))

page_counters: dict[str, int] = {}
points = []
for node, dense_vector, sparse_vector in zip(nodes, dense_vectors, sparse_vectors):
    page_label = node.metadata["page_label"]
    file_name = node.metadata["file_name"]
    chunk_index = page_counters.get(page_label, 0)
    page_counters[page_label] = chunk_index + 1

    point_id = str(uuid5(NAMESPACE_URL, f"{file_name}:{page_label}:{chunk_index}"))
    points.append(
        PointStruct(
            id=point_id,
            vector={
                "dense": dense_vector.tolist(),
                "bm25": SparseVector(
                    indices=sparse_vector.indices.tolist(),
                    values=sparse_vector.values.tolist(),
                ),
            },
            payload={
                "text": node.get_content(),
                "page_label": page_label,
                "file_name": file_name,
                "chunk_index": chunk_index,
            },
        )
    )

client.upsert(collection_name=COLLECTION_NAME, points=points)
print("Points count:", client.get_collection(COLLECTION_NAME).points_count)

Points count: 4


## Retrieval helper

Same hybrid-search-then-rerank recipe as notebooks 08/09, wrapped in a
function so it can run for two different questions without repeating the
cell.

In [3]:
import logging
import os

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("huggingface_hub").setLevel(logging.WARNING)

from fastembed.rerank.cross_encoder import TextCrossEncoder
from qdrant_client.models import FusionQuery, Fusion, Prefetch

reranker = TextCrossEncoder(model_name=RERANK_MODEL)
TOP_K = 2


def retrieve(question: str) -> list[dict]:
    query_dense = next(dense_embedder.embed([question])).tolist()
    query_sparse = next(sparse_embedder.embed([question]))

    hybrid_response = client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=[
            Prefetch(query=query_dense, using="dense", limit=10),
            Prefetch(
                query=SparseVector(
                    indices=query_sparse.indices.tolist(),
                    values=query_sparse.values.tolist(),
                ),
                using="bm25",
                limit=10,
            ),
        ],
        query=FusionQuery(fusion=Fusion.RRF),
        limit=4,
        with_payload=True,
    )
    candidates = hybrid_response.points
    candidate_texts = [c.payload["text"] for c in candidates]
    rerank_scores = list(reranker.rerank(question, candidate_texts))
    scored = sorted(zip(candidates, rerank_scores), key=lambda pair: pair[1], reverse=True)
    return [c.payload for c, _ in scored[:TOP_K]]


grounded_question = "How is citation accuracy measured in this paper?"
ungrounded_question = "What funding source supported this research?"

grounded_chunks = retrieve(grounded_question)
ungrounded_chunks = retrieve(ungrounded_question)

print("Grounded question chunks:")
for chunk in grounded_chunks:
    print(f"  page {chunk['page_label']}: {chunk['text'][:60]}...")

print("\nUngrounded question chunks:")
for chunk in ungrounded_chunks:
    print(f"  page {chunk['page_label']}: {chunk['text'][:60]}...")

Grounded question chunks:
  page 1: Abstract
This paper studies retrieval-augmented generation f...
  page 4: 3. Results
Across a held-out set of research papers, grounde...

Ungrounded question chunks:
  page 4: 3. Results
Across a held-out set of research papers, grounde...
  page 1: Abstract
This paper studies retrieval-augmented generation f...


## Run v1 / v2 / v3 on both questions

Needs `backend/.env` with `OPENAI_API_KEY` (set up in Phase 3). Six real,
billed API calls total.

In [4]:
import logging
import sys

logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("openai").setLevel(logging.WARNING)

sys.path.insert(0, str(Path.cwd().parent))

from src.llm import get_llm
from src.prompts import v1, v2, v3

llm = get_llm("openai")

versions = {"v1": v1, "v2": v2, "v3": v3}
cases = {
    "grounded": (grounded_question, grounded_chunks),
    "ungrounded": (ungrounded_question, ungrounded_chunks),
}

for case_name, (question, case_chunks) in cases.items():
    print(f"########## {case_name.upper()} QUESTION: {question} ##########")
    for version_name, module in versions.items():
        print(f"--- {version_name} ---")
        print(module.answer(question, case_chunks, llm))
        print()

########## GROUNDED QUESTION: How is citation accuracy measured in this paper? ##########
--- v1 ---


2026-11-25 19:13:53,297 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


The context provided does not specify the exact method used to measure citation accuracy in the paper. However, it indicates that citation accuracy is evaluated through a comparison of grounded answers (which include page citations) versus ungrounded answers, with human reviewers rating the trustworthiness of these answers. The improvement in citation accuracy is inferred from these ratings, as grounded answers were deemed more trustworthy than ungrounded ones.

--- v2 ---


2026-11-25 19:13:55,103 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


The context provided does not specify how citation accuracy is measured in the paper. It mentions that combining dense retrieval with a re-ranking stage improves citation accuracy compared to retrieval alone (page 1), but does not detail the methodology or metrics used for measuring citation accuracy. Therefore, based on the given context, an answer cannot be constructed regarding the measurement of citation accuracy.

--- v3 ---


2026-11-25 19:13:56,281 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


... (3 outputs left out)

2026-11-25 19:13:58,856 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


The provided context does not mention any funding source that supported the research. Therefore, I cannot provide an answer based on the information given.

--- v3 ---


2026-11-25 19:14:00,336 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


The context provided does not contain any information about the funding source that supported this research.



## Manual check

Reading all 6 answers above by hand: on the grounded question, do v2/v3
still cite the right pages? On the ungrounded question, does v3 actually
refuse while v1/v2 guess? That observation goes into the learning doc's
"What we did" — honestly, including if a version doesn't behave as
expected.

## Next

Phase 4 (prompts) is done for now. Phase 5 wires the winning prompt
version into a LangGraph workflow (retrieve → grade → generate → verify →
not-found).